# Auditing a care-management risk score

## Section 4 — Investigate what the score tracks

We compare realized medical cost in two ways:

1. within similar narrow commercial-risk ranges; and
2. within the same recorded active chronic-condition count.

These are descriptive comparisons. They can reveal patterns but cannot by
themselves identify every reason that spending differs.


## START/RESTART HERE

Before Task 1, run every setup code cell below in order. Each setup cell is
labeled `# RUN THIS CELL FIRST`; continue until you reach **Task 1**.

The setup cells import packages and load the data.


In [ ]:
# RUN THIS CELL FIRST
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
RACE_STYLES = {
    "black": {"color": "#6A3D9A", "linestyle": "--", "marker": "o"},
    "white": {"color": "#E69F00", "linestyle": "-", "marker": "s"},
}


In [ ]:
# RUN THIS CELL FIRST
DATA_COMMIT = "daceb25bba00e65d7b05882f049e229a8bedb60c"
DATA_URL = (
    "https://gitlab.com/labsysmed/dissecting-bias/-/raw/"
    f"{DATA_COMMIT}/data/data_new.csv"
)
LOCAL_DATA_PATH = Path("data_new.csv")


def load_data(required_columns):
    """Load the course data from a local copy or the pinned public URL."""
    source = LOCAL_DATA_PATH if LOCAL_DATA_PATH.is_file() else DATA_URL
    data = pd.read_csv(source)

    missing_columns = sorted(set(required_columns) - set(data.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    return data


In [ ]:
# RUN THIS CELL FIRST
section_4_columns = ["risk_score_t", "cost_t", "gagne_sum_t", "race"]
df = load_data(section_4_columns)


## Variables used in this section

Each row represents one patient observed in one year.

- `risk_score_t`: the existing commercial score for year t. Larger values rank
  a row as higher commercial risk.
- `cost_t`: total realized medical cost in year t, measured in dollars.
- `gagne_sum_t`: the number of active chronic conditions recorded in year t.
  It is a recorded health-status proxy, not a complete measure of health need.
- `race`: the self-reported audit group, `black` or `white`.

All percentiles and score ranges below are pooled across race so the two groups
are compared on the same commercial-score scale.


## Task 1 — Compare future cost within narrow commercial-risk ranges

### Provided score-range setup from Section 3

Tasks 1A–1D were completed in Section 3. The complete code below recreates
the pooled commercial-risk percentiles and 100 narrow score ranges while
retaining `cost_t` for this section. Run the provided cell, then begin your
work at Task 1E.


In [ ]:
# PROVIDED: score-range setup completed in Section 3
audit = df.loc[:, section_4_columns].copy()

audit["risk_percentile"] = (
    audit["risk_score_t"].rank(method="average", pct=True) * 100
)

stable_rank = audit["risk_score_t"].rank(method="first")
audit["risk_bin"] = (
    pd.qcut(stable_rank, q=100, labels=False) + 1
)

bin_sizes = audit["risk_bin"].value_counts()
number_of_bins = audit["risk_bin"].nunique()
smallest_bin = bin_sizes.min()
largest_bin = bin_sizes.max()

number_of_bins, smallest_bin, largest_bin


### 1E — Build `cost_by_risk`

Create a pandas DataFrame named `cost_by_risk`. Each row should summarize the
patient-year rows that share one `risk_bin` and one `race`; it should not
represent an individual patient-year.

Both races occur in all 100 bins, so the finished DataFrame should have 200
rows and these five columns:

- `risk_bin`: the score-range number;
- `race`: the audit group;
- `risk_percentile`: the arithmetic mean pooled percentile in that group;
- `mean_cost`: the arithmetic mean `cost_t`, in dollars, in that group; and
- `rows`: the number of patient-year rows contributing to that summary row.

The template loops over bins and races and selects each group with Boolean
conditions. Replace the three `...` placeholders with the required
calculations, then convert `cost_by_risk_rows` to a DataFrame.


In [ ]:
cost_by_risk_rows = []

for bin_number in range(1, 101):
    for race_name in ["black", "white"]:
        in_group = (
            (audit["risk_bin"] == bin_number)
            & (audit["race"] == race_name)
        )
        group_rows = audit.loc[in_group]

        cost_by_risk_rows.append(
            {
                "risk_bin": bin_number,
                "race": race_name,
                "risk_percentile": ...,
                "mean_cost": ...,
                "rows": ...,
            }
        )

cost_by_risk = pd.DataFrame(cost_by_risk_rows)
cost_by_risk.head()


### 1F — Sort and check `cost_by_risk`

Sort `cost_by_risk` first by `race` and then by `risk_percentile`. Confirm that
it has shape `(200, 5)` and that its `rows` column sums to all 48,784
patient-year rows. Display its first five rows.


In [ ]:
# TODO


### 1G — Plot the comparison

The code cell immediately below is complete plotting code provided by the
instructor. Do not edit it. Run it after constructing and sorting
`cost_by_risk`.

The cell requires the five columns defined in 1E. Each point represents the
arithmetic mean future cost for one score-range and race group, not an
individual patient-year. The horizontal axis is mean pooled `risk_percentile`;
the vertical axis is `mean_cost`. Sorting matters because the code connects
rows in their current order.

The vertical axis uses a logarithmic display scale because the group means span
a wide range. The costs and their arithmetic means have not been
log-transformed.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for race, style in RACE_STYLES.items():
    plot_data = cost_by_risk.loc[cost_by_risk["race"] == race]
    ax.plot(
        plot_data["risk_percentile"],
        plot_data["mean_cost"],
        label=race.title(),
        linewidth=2,
        markersize=3,
        markevery=5,
        **style,
    )

ax.set_yscale("log")
ax.set(
    xlabel="Commercial-risk percentile (pooled)",
    ylabel="Mean future cost in dollars (log display scale)",
    title="Future cost across commercial-risk percentiles",
)
ax.legend(title="Audit group")
plt.show()


**Read the figure:** Compare the vertical positions of the dashed-circle Black
series and solid-square White series across the score distribution. Where are
the series broadly similar, and where are the synthetic-data tails noisy?

**Your observation:** [Write one or two sentences here.]


## Task 2 — Compare future cost at the same recorded condition count

### 2A — Pool high condition counts and count rows

Before counting rows, create `condition_group` by keeping condition counts 0
through 6 unchanged and pooling every `gagne_sum_t` value of 7 or more into
one group represented by the value 7. In the table and plot, 7 represents 7+.

Create the following objects:

- `condition_levels`: an ascending list of the distinct `condition_group` values;
  and
- `condition_counts`: an 8 × 2 pandas DataFrame. Each row represents one
  condition group, its index is `condition_group`, and its `black`
  and `white` columns contain patient-year row counts.

The template supplies the loop and DataFrame construction. Replace the two
`...` placeholders with Boolean row counts.


In [ ]:
audit["condition_group"] = audit["gagne_sum_t"].clip(upper=7)
condition_levels = sorted(audit["condition_group"].unique().tolist())
condition_count_rows = []

for condition_level in condition_levels:
    at_level = audit["condition_group"] == condition_level

    black_rows = ...
    white_rows = ...

    condition_count_rows.append(
        {
            "condition_group": condition_level,
            "black": black_rows,
            "white": white_rows,
        }
    )

condition_counts = pd.DataFrame(condition_count_rows).set_index("condition_group")
condition_counts.columns.name = "race"
condition_counts


### 2B — Build `cost_by_conditions`

Create a pandas DataFrame named `cost_by_conditions`. Each row should summarize
one `condition_group` and `race` combination. Because there are eight groups
for both races, the finished DataFrame should have 16 rows and these four
columns:

- `condition_group`: the recorded active chronic-condition count, with 7
  representing 7 or more;
- `race`: the audit group;
- `mean_cost`: the arithmetic mean `cost_t`, in dollars, in that group; and
- `rows`: the number of patient-year rows contributing to that summary row.

The template supplies the loops and Boolean filters. Replace the two `...`
placeholders with the required calculations.


In [ ]:
cost_by_condition_rows = []

for condition_level in condition_levels:
    for race_name in ["black", "white"]:
        in_group = (
            (audit["condition_group"] == condition_level)
            & (audit["race"] == race_name)
        )
        group_rows = audit.loc[in_group]

        cost_by_condition_rows.append(
            {
                "condition_group": condition_level,
                "race": race_name,
                "mean_cost": ...,
                "rows": ...,
            }
        )

cost_by_conditions = pd.DataFrame(cost_by_condition_rows)
cost_by_conditions.head()


### 2C — Sort and check `cost_by_conditions`

Sort `cost_by_conditions` first by `race` and then by `condition_group`. Confirm
that it has shape `(16, 4)` and that its `rows` column sums to all 48,784 rows
in `audit`. Display the completed table.


In [ ]:
# TODO


### 2D — Plot the comparison

The code cell immediately below is complete plotting code provided by the
instructor. Do not edit it. Run it after constructing and sorting
`cost_by_conditions`.

Each point represents arithmetic mean future cost for one pooled recorded
condition-count and race group. The horizontal axis is `condition_group`; the
vertical axis is `mean_cost` on a logarithmic display scale. The two lines
therefore compare racial groups at the same recorded condition count, with all
counts of 7 or more pooled at the final point. Sorting
matters because the plotting code connects rows in their current order.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for race, style in RACE_STYLES.items():
    plot_data = cost_by_conditions.loc[cost_by_conditions["race"] == race]
    ax.plot(
        plot_data["condition_group"],
        plot_data["mean_cost"],
        label=race.title(),
        linewidth=2,
        markersize=5,
        **style,
    )

ax.set_yscale("log")
ax.set_xticks(range(8), ["0", "1", "2", "3", "4", "5", "6", "7+"])
ax.set(
    xlabel="Recorded active chronic-condition count (7+ pooled)",
    ylabel="Mean future cost in dollars (log display scale)",
    title="Future cost at the same recorded condition count",
)
ax.legend(title="Audit group")
plt.show()


**Read the figure:** At the same supported condition count, compare the
dashed-circle Black series with the solid-square White series. Describe the
broad pattern.


## END-OF-SECTION CHECKPOINT

Use both figures to answer:

1. What outcome does the commercial score appear to track?
2. At the same recorded active chronic-condition count, how does mean realized
   cost compare by race?
3. What problem can arise if realized cost is used to allocate care intended
   for health need?

**Your interpretation:** [Write two or three sentences here.]
